This notebook makes sure that the whole Fisher pipeline runs smoothly and prepares the 3d plotting code needed for our money plot.

In [1]:
from pathlib import Path
import jax.numpy as np

from gwfast.gwfastGlobals import detectors as det_dict, detPath
import gwfast.waveforms as waveforms
from gwfast.detector import Detector
from gwfast.signals import AGNLensedGWSignal
import gwfast.network as network
from gwfast.fisherTools import reduce_Fisher_matrix, CovMatr, plot_corners

ModuleNotFoundError: No module named 'jax'

## Test Fisher (with 3d input)

In [ ]:
# Set up detectors
H1 = Detector('H1', **det_dict['H1'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/AplusDesign.txt')
L1 = Detector('L1', **det_dict['L1'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/AplusDesign.txt')
V1 = Detector('V1', **det_dict['Virgo'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/avirgo_O5low_NEW.txt')

wf_model = waveforms.IMRPhenomD()

H1_AGN = AGNLensedGWSignal(wf_model=wf_model, detector=H1, fmin=10)
L1_AGN = AGNLensedGWSignal(wf_model=wf_model, detector=L1, fmin=10)
V1_AGN = AGNLensedGWSignal(wf_model=wf_model, detector=V1, fmin=10)
HLV_AGN = network.DetNet({'H1': H1_AGN, 'L1': L1_AGN, 'V1': V1_AGN})

Initializing jax...
Jax local device count: 1
Jax device count: 1
Initializing jax...
Jax local device count: 1
Jax device count: 1
Initializing jax...
Jax local device count: 1
Jax device count: 1


In [ ]:
# Define a benchmark event
n = 2
shape = (n, n, n)
events = {
    'Mc':np.full(shape, 30), 'eta':np.full(shape, 0.24), 
    'chi1z':np.full(shape, 0.3), 'chi2z':np.full(shape, 0.5), 
    'tcoal':np.full(shape, 0), 'phase':np.full(shape, 2), 
    'R_orbit':np.full(shape, 100), 'M_lz':np.full(shape, 1e4), 'src_pos':np.full(shape, 0.5),
    'iota':np.full(shape, 0.99*np.pi/2), 'psi':np.full(shape, 4), 
    'dL':np.full(shape, 0.8), 'theta':np.full(shape, 1.87), 'phi':np.full(shape, 2.66), 
}
events = {key: val.astype(np.float64) for key, val in events.items()}

# Sample parameter space
R_orbit_array = np.geomspace(20, 5000, n)
src_pos_array = np.linspace(0.01, 0.99, n)
dL_array = np.geomspace(0.1, 10, n)

# Make grid
R_orbit_mesh, src_pos_mesh, dL_mesh = np.meshgrid(R_orbit_array, src_pos_array, dL_array, indexing='xy')
events['R_orbit'] = R_orbit_mesh
events['src_pos'] = src_pos_mesh
events['dL'] = dL_mesh

In [ ]:
import importlib
import gwfast.fisherTools
importlib.reload(gwfast.fisherTools)
from gwfast.fisherTools import reduce_Fisher_matrix, CovMatr, plot_corners

In [ ]:
fisher_AGN = HLV_AGN.FisherMatr(events, res=1000)
keys = list(events.keys()).copy()
reduced_fisher_AGN, _ = reduce_Fisher_matrix(fisher_AGN, keys=keys)
reduced_cov_AGN, ie = CovMatr(reduced_fisher_AGN)

Computing Fisher for H1...
Computing Fisher for L1...
Computing Fisher for V1...
Done.


## Test 3d plotting isosurface

In [1]:
import numpy as onp
from skimage.measure import marching_cubes
from scipy.interpolate import interp1d
import pyvista as pv

In [2]:
n = 50
R_orbit_array = onp.geomspace(20, 5000, n)
src_pos_array = onp.linspace(0.01, 0.99, n)
dL_array = onp.geomspace(0.1, 10, n) # Gpc

R_orbit_mesh, src_pos_mesh, dL_mesh = onp.meshgrid(R_orbit_array, src_pos_array, dL_array, indexing='xy')
M_lens = 1e5 # M_sun
M_lens_mesh = onp.full(dL_mesh.shape, M_lens)
delta_mesh = 4.785e-20 * M_lens_mesh / (dL_mesh * 10)
theta_E = delta_mesh * onp.sqrt((2 * R_orbit_mesh * onp.sqrt(1 - src_pos_mesh ** 2)) / 
                                         (1 + delta_mesh * R_orbit_mesh * onp.sqrt(1 - src_pos_mesh ** 2)))
theta_E_in_arcsec = theta_E / onp.pi * 180 * 3600
print(onp.max(theta_E_in_arcsec), onp.min(theta_E_in_arcsec), onp.average(theta_E_in_arcsec), onp.median(theta_E_in_arcsec))

9.869524225277581e-08 2.3445005559330573e-11 6.545151124762322e-09 2.133393460611529e-09


In [ ]:
n_point = 50
r_min, r_max = 20, 5000  # R_Sch
y_min, y_max = 0.01, 0.99  # R_orbit
d_min, d_max = 10, 1000  # Mpc

log_r_min, log_r_max = np.log10(r_min), np.log10(r_max)
log_r_array = onp.linspace(log_r_min, log_r_max, n_point)
y_array = onp.linspace(y_min, y_max, n_point)
log_dL_min, log_dL_max = np.log10(d_min), np.log10(d_max)  # log(Mpc)
log_dL_array = onp.linspace(log_dL_min, log_dL_max, n_point)

log_r_mesh, y_mesh, log_dL_mesh = \
    onp.meshgrid(log_r_array, y_array, log_dL_array, indexing='xy')
r_mesh = onp.power(10, log_r_mesh)
dL_mesh = onp.power(10, log_dL_mesh)  # Mpc
M_lens = 1e5 # M_sun
delta_mesh = 4.785e-20 * M_lens / dL_mesh
theta_E = delta_mesh * onp.sqrt((2 * r_mesh * onp.sqrt(1 - y_mesh ** 2)) / 
                                         (1 + delta_mesh * r_mesh * onp.sqrt(1 - y_mesh ** 2)))
theta_E_in_arcsec = theta_E / onp.pi * 180 * 3600

print(onp.max(theta_E_in_arcsec), onp.min(theta_E_in_arcsec), 
      onp.average(theta_E_in_arcsec), onp.median(theta_E_in_arcsec))

def plot_isosurface(plotter, x_array, y_array, z_array, 
                    volume, level, label, color, opacity=0.7,
                    xscale=1, yscale=1, zscale=1):
    dx = x_array[1] - x_array[0]
    dy = y_array[1] - y_array[0]
    dz = z_array[1] - z_array[0]
    
    verts, faces, _, _ = marching_cubes(
        volume, level=level, spacing=(dx, dy, dz),
    )
    # Convert back to linear scale for plotting
    verts[:, 0] = onp.power(10, verts[:, 0] + x_array.min()) * xscale
    verts[:, 1] *= yscale
    verts[:, 2] = onp.power(10, verts[:, 2] + z_array.min()) * zscale

    # print(verts[:, 0], verts[:, 0].max())
    # print(verts[:, 1], verts[:, 1].max())
    # print(verts[:, 2], verts[:, 2].max())

    faces_pv = onp.hstack([[3, *face] for face in faces])
    mesh = pv.PolyData(verts, faces_pv)
    actor = plotter.add_mesh(mesh, color=color, opacity=opacity, show_edges=True, label=label)
    return mesh, actor

9.869524225383833e-09 2.3445005559330574e-12 6.545151124773786e-10 2.1333934606118457e-10


In [204]:
theta_E_levels = [1e-11, 1e-10, 1e-9]
labels = ['theta_E = ' + str(i) + ' arcsec' for i in theta_E_levels]
colors = ['midnightblue', 'teal', 'cadetblue']

xscale = 5000 
zscale = 1000

plotter = pv.Plotter()
for i in range(3):
    mesh, actor = plot_isosurface(
        plotter, log_r_array, y_array, log_dL_array,
        theta_E_in_arcsec, theta_E_levels[i], labels[i], colors[i], 
        xscale=1/xscale, zscale=1/zscale
    )

plotter.show_bounds(
    xtitle=f'R_orbit / ({xscale:d} R_Sch)',
    ytitle='y / R_orbit',
    ztitle=f'dL / Gpc',
    grid='front',
    location='outer',
    all_edges=True
)

plotter.add_legend(
    size=(0.2, 0.1),
    loc='lower left'
)

plotter.show()

[0.004      0.00447711 0.00447711 ... 1.         0.98466691 1.        ] 1.0000000000000122
[0.         0.         0.00054723 ... 0.5        0.5        0.51478951] 0.5147895050048827
[0.62422951 0.62411024 0.62505519 ... 0.95993444 1.         1.        ] 1.000000000000005
[0.004      0.00447711 0.00447711 ... 0.98075236 0.94473257 0.90668669] 1.0000000000000122
[0.        0.        0.0171679 ... 0.98      0.98      0.98     ] 0.9799999999999999
[0.06248927 0.06247675 0.06551286 ... 0.39069399 0.42919343 0.47148664] 0.9874690579977101
[0.00447711 0.004      0.004      ... 1.         0.96299757 0.92596102] 1.0000000000000122
[0.16723417 0.18       0.16716368 ... 0.98       0.98       0.98      ] 0.9799999999999999
[0.01       0.01037526 0.01       ... 0.03707877 0.04094915 0.04498433] 0.09879618058309306


Widget(value='<iframe src="http://localhost:50507/index.html?ui=P_0x5fb4b9400_124&reconnect=auto" class="pyvis…

In [190]:
mesh

PolyData,Information
N Cells,6632
N Points,3431
N Strips,0
X Bounds,"2.000e-02, 5.000e+00"
Y Bounds,"1.672e-01, 9.800e-01"
Z Bounds,"1.000e-02, 9.880e-02"
N Arrays,0


## Alternative approach?


In [280]:
n_point = 50
r_min, r_max = 20, 5000  # R_Sch
y_min, y_max = 0.01, 0.99  # R_orbit
d_min, d_max = 10, 1000  # Mpc
xscale = 5000
zscale = 1000

grid = pv.ImageData(
    dimensions=(n_point, n_point, n_point),
    spacing=(
        (r_max - r_min) / (n_point - 1) / xscale, 
        (y_max - y_min) / (n_point - 1), 
        (d_max - d_min) / (n_point - 1) / zscale
    ),
    origin=(r_min / xscale, y_min, d_min / zscale)
)

r_mesh, y_mesh, d_mesh = grid.points.T
r_mesh *= xscale
d_mesh *= zscale  # Convert to Gpc

delta_mesh = 4.785e-20 * M_lens / d_mesh
theta_E = delta_mesh * onp.sqrt((2 * r_mesh * onp.sqrt(1 - y_mesh ** 2)) / 
                                         (1 + delta_mesh * r_mesh * onp.sqrt(1 - y_mesh ** 2)))
theta_E_in_arcsec = theta_E / onp.pi * 180 * 3600

out = grid.contour(
    [1e-11, 1e-10, 1e-9],
    scalars=theta_E_in_arcsec,
    method='flying_edges',
)

plotter = pv.Plotter()
plotter.enable_anti_aliasing('ssaa', multi_samples=32)
plotter.add_mesh(out, log_scale=True, opacity=0.7, show_edges=True, 
                 scalar_bar_args={'vertical': True, 
                                  'title': 'theta_E / arcsec',
                                  'position_x': 0.85})

plotter.show_bounds(
    xtitle=f'R_orbit / ({xscale:d} R_Sch)',
    ytitle='y / R_orbit',
    ztitle=f'dL / Gpc',
    grid='front',
    location='outer',
    all_edges=True
)

plotter.show()

Widget(value='<iframe src="http://localhost:50507/index.html?ui=P_0x64aca2af0_170&reconnect=auto" class="pyvis…

In [ ]:
n_point = 50
r_min, r_max = 20, 5000  # R_Sch
y_min, y_max = 0.01, 0.99  # R_orbit
d_min, d_max = 10, 1000  # Mpc
xscale = 5000
zscale = 1000

grid = pv.ImageData(
    dimensions=(n_point, n_point, n_point),
    spacing=(
        (r_max - r_min) / (n_point) / xscale, 
        (y_max - y_min) / (n_point), 
        (d_max - d_min) / (n_point) / zscale
    ),
    origin=(r_min / xscale, y_min, d_min / zscale)
)

r_mesh, y_mesh, d_mesh = grid.points.T
r_mesh *= xscale
d_mesh *= zscale  # Convert to Gpc

delta_mesh = 4.785e-20 * M_lens / d_mesh
theta_E = delta_mesh * onp.sqrt((2 * r_mesh * onp.sqrt(1 - y_mesh ** 2)) / 
                                         (1 + delta_mesh * r_mesh * onp.sqrt(1 - y_mesh ** 2)))
theta_E_in_arcsec = theta_E / onp.pi * 180 * 3600

grid.point_data['values'] = theta_E_in_arcsec
slices = grid.slice_orthogonal(x=0.7, y=0.3, contour=True, )

plotter = pv.Plotter()
plotter.enable_anti_aliasing('ssaa', multi_samples=32)
plotter.add_mesh(slices, log_scale=True, opacity=0.7, show_edges=True, 
                 scalar_bar_args={'vertical': True, 
                                  'title': 'theta_E / arcsec',
                                  'position_x': 0.85})

for slice_i in slices:
    contours = slice_i.contour(
        [1e-9, 5e-10, 1e-10, 5e-11],
        scalars='values',
        method='contour',
    )
    plotter.add_mesh(contours, color="white", opacity=0.9, line_width=2)

plotter.show_bounds(
    xtitle=f'R_orbit / ({xscale:d} R_Sch)',
    ytitle='y / R_orbit',
    ztitle=f'dL / Gpc',
    grid='front',
    location='outer',
    all_edges=True
)

plotter.show()

Widget(value='<iframe src="http://localhost:50507/index.html?ui=P_0x6a6ff6280_169&reconnect=auto" class="pyvis…

## In fact, why have one when you can have two :)

In [ ]:
import numpy as np
import pyvista as pv

: 

In [ ]:
n_point = 50
r_min, r_max = 20, 5000  # R_Sch
y_min, y_max = 0.01, 0.99  # R_orbit
d_min, d_max = 10, 1000  # Mpc
xscale = 5000
zscale = 1000
M_lens = 1e5  # M_sun

grid = pv.ImageData(
    dimensions=(n_point, n_point, n_point),
    spacing=(
        (r_max - r_min) / (n_point) / xscale, 
        (y_max - y_min) / (n_point), 
        (d_max - d_min) / (n_point) / zscale
    ),
    origin=(r_min / xscale, y_min, d_min / zscale)
)

r_mesh, y_mesh, d_mesh = grid.points.T
r_mesh *= xscale
d_mesh *= zscale  # Convert to Gpc

delta_mesh = 4.785e-20 * M_lens / d_mesh
theta_E = delta_mesh * np.sqrt((2 * r_mesh * np.sqrt(1 - y_mesh ** 2)) / 
                                         (1 + delta_mesh * r_mesh * np.sqrt(1 - y_mesh ** 2)))
theta_E_in_arcsec = theta_E / np.pi * 180 * 3600
log_theta_E = np.log10(theta_E_in_arcsec)

bound_kwargs = dict(
    xtitle=f'R_orbit / ({xscale:d} R_Sch)',
    ytitle='y / R_orbit',
    ztitle=f'dL / Gpc',
    grid='front',
    location='outer',
    all_edges=True
)

plotter = pv.Plotter(shape='1|1', off_screen=True, notebook='local')
plotter.link_views()

plotter.subplot(0)
out = grid.contour(
    [-11, -10, -9],
    scalars=log_theta_E,
    method='flying_edges',
)
plotter.add_mesh(out, opacity=0.7, 
                 scalar_bar_args={'vertical': True, 
                                  'title': 'log10(theta_E / arcsec)',
                                  'position_x': 0.85})
plotter.show_bounds(**bound_kwargs)

plotter.subplot(1)
grid.point_data['values'] = log_theta_E
slices = grid.slice_orthogonal(x=0.7, y=0.3, contour=True)
plotter.add_mesh(slices, opacity=0.7,
                 scalar_bar_args={'vertical': True, 
                                  'title': 'log10(theta_E / arcsec)',
                                  'position_x': 0.85})

for slice_i in slices:
    contours = slice_i.contour(
        np.log10(np.array([1e-9, 5e-10, 1e-10, 5e-11])),
        scalars='values',
        method='contour',
    )
    plotter.add_mesh(contours, color="white", opacity=0.9, line_width=2)

plotter.show_bounds(**bound_kwargs)

## Note!!!
## Either use other anti-aliasing method or use the remote rendering mode
plotter.enable_anti_aliasing('ssaa', multi_samples=32)

plotter.show(jupyter_backend='trame')

/users/hin-wai.leong/.conda/envs/180125_py311_AGNLensing/lib/python3.11/site-packages/pyvista/plotting/plotter.py:162: UserWarning: 
This system does not appear to be running an xserver.
PyVista will likely segfault when rendering.

Alternatively, an offscreen version using OSMesa libraries and ``vtk-osmesa`` is available.

  warnings.warn(
ERROR:root:bad X server connection. DISPLAY=
2025-06-12 11:10:06.593 (   1.033s) [    7FFB744AC400]vtkXOpenGLRenderWindow.:456    ERR| vtkXOpenGLRenderWindow (0x562e5efcdb70): bad X server connection. DISPLAY=


In [264]:
slices[0].point_data['values'].max()

3.118561916395323e-09

In [241]:
r_mesh.shape

(132651,)

In [237]:
theta_E_in_arcsec.shape

(125000,)

In [234]:
grid

ImageData (0x650676d00)
  N Cells:      117649
  N Points:     125000
  X Bounds:     4.000e-03, 1.000e+00
  Y Bounds:     1.000e-02, 9.900e-01
  Z Bounds:     1.000e-02, 1.000e+00
  Dimensions:   50, 50, 50
  Spacing:      2.033e-02, 2.000e-02, 2.020e-02
  N Arrays:     1

In [ ]:
## Failed attempts:

    # actor.SetScale(1, 
    #                 R_orbit_array.ptp()/src_pos_array.ptp(), 
    #                 R_orbit_array.ptp()/dL_array.ptp(),
    # )
                    # reset_camera=True)
# r_range = R_orbit_array.max() - R_orbit_array.min()
# src_pos_range = src_pos_array.max() - src_pos_array.min()
# dL_range = dL_array.max() - dL_array.min()
# plotter.set_scale(xscale=1/, 
#                   yscale=1, 
#                   zscale=1/10)
# plotter.show_grid()
# plotter.add_point_labels()

# plotter.set_scale(xscale=dL_range/r_range, 
#                   yscale=dL_range/src_pos_range, 
#                   zscale=1)
# n_labels = 5
# interval = 1 / (n_labels - 1)
# ticks = [i * interval for i in range(n_labels)]
# input_idx_interval = n / (n_labels - 1)

# x_ticks = ticks
# x_labels = [int(round(R_orbit_array[int(i * (input_idx_interval - 0.25))], 0)) for i in range(n_labels)]
# y_ticks = ticks
# y_labels = [src_pos_array[int(i * (input_idx_interval - 0.25))] for i in range(n_labels)]
# z_ticks = ticks
# z_labels = [round(dL_array[int(i * (input_idx_interval - 0.25))], 2) for i in range(n_labels)]

# x_points = onp.array([[x, 0, 0] for x in x_ticks])
# y_points = onp.array([[0, y, 0] for y in y_ticks])
# z_points = onp.array([[0, 0, z] for z in z_ticks])

# plotter.add_point_labels(x_points, x_labels, font_size=12, point_size=0)
# plotter.add_point_labels(y_points, y_labels, font_size=12, point_size=0)
# plotter.add_point_labels(z_points, z_labels, font_size=12, point_size=0)



# plotter.show_bounds(
#     mesh,
#     bounds=[R_orbit_array.min(), R_orbit_array.max(),
#                   src_pos_array.min(), src_pos_array.max(),
#                   dL_array.min(), dL_array.max()],
#     grid='front',
#     location='outer',
#     all_edges=True,
#     xtitle='R_orbit',
#     ytitle='y',
#     ztitle='dL',
#     n_xlabels=10
# )
# ruler = plotter.add_ruler(
#     pointa=(R_orbit_array.min() * dL_range/r_range, 0, 0),
#     pointb=(R_orbit_array.max() * dL_range/r_range, 0, 0),
#     flip_range=True,
#     title='R_orbit (R_Sch)',
#     scale=dL_range/r_range,
# )
# # a, b = ruler.GetRange()  # Either (0, distance) or (distance, 0)
# # ruler.SetRange(a / r_range *dL_range, b / r_range*dL_range)
# ruler = plotter.add_ruler(
#     pointa=(0, src_pos_array.min() * src_pos_range, 0),
#     pointb=(0, src_pos_array.max() * src_pos_range, 0),
#     title='y_src (R_orbit)',
# )
# # a, b = ruler.GetRange()  # Either (0, distance) or (distance, 0)
# # ruler.SetRange(a / src_pos_range, b / src_pos_range)
# ruler = plotter.add_ruler(
#     pointa=(0, 0, dL_array.min()),
#     pointb=(0, 0, dL_array.max()),
#     flip_range=True,
#     title='d_L (Gpc)',
# )
# a, b = ruler.GetRange()  # Either (0, distance) or (distance, 0)
# ruler.SetRange(a / dL_range, b / dL_range)

In [117]:
actor.SetScale?

Docstring:
SetScale(self, x:float, y:float, z:float) -> None
C++: virtual void SetScale(double x, double y, double z)
SetScale(self, scale:[float, float, float]) -> None
C++: virtual void SetScale(double scale[3])
SetScale(self, s:float) -> None
C++: void SetScale(double s)

Set/Get the scale of the actor. Scaling in performed
independently on the X, Y and Z axis. A scale of zero is illegal
and will be replaced with one.
Type:      builtin_function_or_method


In [193]:
plotter.show_bounds?

Signature:
plotter.show_bounds(
    mesh=None,
    bounds=None,
    axes_ranges=None,
    show_xaxis=True,
    show_yaxis=True,
    show_zaxis=True,
    show_xlabels=True,
    show_ylabels=True,
    show_zlabels=True,
    bold=True,
    font_size=None,
    font_family=None,
    color=None,
    xtitle='X Axis',
    ytitle='Y Axis',
    ztitle='Z Axis',
    n_xlabels=5,
    n_ylabels=5,
    n_zlabels=5,
    use_2d=False,
    grid=None,
    location='closest',
    ticks=None,
    all_edges=False,
    corner_factor=0.5,
    fmt=None,
    minor_ticks=False,
    padding=0.0,
    use_3d_text=True,
    render=None,
    **kwargs,
)
Docstring:
Add bounds axes.

Shows the bounds of the most recent input mesh unless mesh is
specified.

Parameters
----------
mesh : pyvista.DataSet | pyvista.MultiBlock, optional
    Input mesh to draw bounds axes around.

bounds : sequence[float], optional
    Bounds to override mesh bounds in the form ``[xmin, xmax,
    ymin, ymax, zmin, zmax]``.

axes_ranges : seq

In [107]:
import numpy as np
# Define a simple linear surface
x = np.array([1,2,3,4,5,6,7,8,9])
y = np.array([1,2,3,4,5,6,7,8,9])
x, y = np.meshgrid(x, y)
z = x*y

# Create and plot structured grid
grid = pv.StructuredGrid(x, y, z)
plotter = pv.Plotter()
plotter.add_mesh(grid, scalars=grid.points[:, -1], show_edges=True,
                 scalar_bar_args={'vertical': True})
plotter.show_grid()
# scale plot to enforce 1:1:1 aspect ratio
# plotter.set_scale(xscale=1, yscale=x.ptp()/y.ptp(), zscale=x.ptp()/z.ptp())
plotter.show()

/Users/Samson/.conda/envs/IGWN_py39_clone_2/lib/python3.9/site-packages/pyvista/core/utilities/points.py:77: UserWarning: Points is not a float type. This can cause issues when transforming or applying filters. Casting to ``np.float32``. Disable this by passing ``force_float=False``.
  warnings.warn(


Widget(value='<iframe src="http://localhost:50507/index.html?ui=P_0x5035c6c70_75&reconnect=auto" class="pyvist…

In [108]:
pv.__version__

'0.45.2'

## Test PyVista in cluster

In [1]:
import os
os.environ['DISPLAY'] = ':99.0'
os.environ['PYVISTA_OFF_SCREEN'] = 'true'

import numpy as np
import pyvista as pv
pv.start_xvfb()

pv.set_plot_theme('document')

# pv.set_jupyter_backend('static')

pl = pv.Plotter(off_screen=True)
pl.add_mesh(pv.ParametricKlein())
pl.show()

/users/hin-wai.leong/.conda/envs/180125_py311_AGNLensing/lib/python3.11/site-packages/pyvista/plotting/utilities/xvfb.py:48: PyVistaDeprecationWarning: This function is deprecated and will be removed in future version of PyVista. Use vtk-osmesa instead.
  warnings.warn(
ERROR:root:Could not find a decent config
2025-06-12 14:35:38.831 (   4.174s) [    7F74621A4400]vtkXOpenGLRenderWindow.:256    ERR| vtkXOpenGLRenderWindow (0x55bea7d83430): Could not find a decent config



: 

In [1]:
import vtk; vtk.__version__

'9.3.1'